# RedZone world model: scaling ladder on a Kaggle TPU v5e-8 (EXP-106 throughput + EXP-107 ladder + EXP-108 MoE)

Personal use only: plan B data, so every grades.json carries "personal-use, not for commercial release".
One portable trainer (scripts/nfm_train.py: torch_xla SPMD over the 8 chips, bf16, sequence packing, RoPE) trains the
dense ladder 18m -> 60m -> 150m -> 300m on the same 1.2B-token budget, then MoE vs dense at 60m. Each point is graded per
league on held-out nights (locked and scorecard nights are never downloaded). Data: public play by play from GitHub
(nflverse NFL, NBA.com cdn NBA, Retrosheet MLB), tokenized with vocab v2 (~60 tokens per NFL play).

Settings > Accelerator > **TPU VM v5e-8**, Internet **on**, then **Run all**. About 3.5 h of the weekly TPU quota (data build
~40 min, ladder ~2.5 h, MoE ~0.4 h). Checkpoints go to /kaggle/working every 30 min: if the session ends, Run all again and
every point resumes where it stopped (finished points are skipped). The last cell prints the ladder table.


In [ ]:
PIN = "bd1f807883a23294b7905c0343f33e1c77da4cb8"   # kaggle-inputs commit: factory code pack (nfm_train.py + noggin_tokens_v2.py)
REPO = "BouncySlime1215/redzone-everything"
LEAGUES = "nfl,nba,mlb"
LADDER = ["18m", "60m", "150m", "300m", "moe-60m"]
EXPS = {"moe-60m": "EXP-108"}
MINUTES = {"18m": 20, "60m": 30, "150m": 50, "300m": 90, "moe-60m": 40}   # caps; the token budget ends most runs first
CTX, BATCH, TRAIN_TOKENS = 2048, 64, 1.2e9   # 64 sequences = 8 per chip
RUN, CODE = "/kaggle/working/rz", "/kaggle/working/rz/code"
ENV = {"PJRT_DEVICE": "TPU"}


In [ ]:
# torch_xla: Kaggle's TPU image usually ships it; if not, install the wheel that matches the installed torch
try:
    import torch_xla, torch
    print("torch", torch.__version__, "torch_xla", torch_xla.__version__)
except Exception as e:
    print("installing torch_xla:", type(e).__name__)
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch~=2.6.0", "torch_xla[tpu]~=2.6.0",
                    "-f", "https://storage.googleapis.com/libtpu-releases/index.html"], check=True)
    import torch_xla, torch
    print("torch", torch.__version__, "torch_xla", torch_xla.__version__)
import torch_xla.runtime as xr
print("TPU chips:", xr.global_runtime_device_count())


In [ ]:
import json, os, subprocess, sys, tarfile, time, urllib.request
T_START = time.time()
os.makedirs(CODE, exist_ok=True)
if not os.path.exists(f"{CODE}/rz/scripts/nfm_train.py"):
    for i, ref in enumerate((PIN, PIN[:12], PIN, PIN[:12])):     # the raw CDN can lag a fresh push for a minute
        url = f"https://raw.githubusercontent.com/{REPO}/{ref}/factory/rz_factory_code.tar.gz"
        try:
            print("downloading", url)
            urllib.request.urlretrieve(url, f"{RUN}/code.tar.gz")
            break
        except Exception as e:
            print(" ", type(e).__name__, e)
            time.sleep(30 * i)
    else:
        raise SystemExit("could not download the code pack; Run all again in a minute")
    with tarfile.open(f"{RUN}/code.tar.gz") as t:
        t.extractall(CODE, filter="data")
print("code:", sorted(os.listdir(f"{CODE}/rz")))


In [ ]:
def run(preset, extra=()):
    out = f"{RUN}/nfm/{preset}"
    os.makedirs(out, exist_ok=True)
    if os.path.exists(f"{out}/grades.json"):
        print(preset, "done already"); return
    cmd = [sys.executable, "-u", f"{CODE}/rz/scripts/nfm_train.py", "--out", out, "--ckpt-dir", out,
           "--data-dir", f"{RUN}/data", "--build-vocab", "2", "--leagues", LEAGUES, "--preset", preset,
           "--ctx", str(CTX), "--batch", str(BATCH), "--train-tokens", str(TRAIN_TOKENS),
           "--minutes", str(MINUTES[preset]), "--exp", EXPS.get(preset, "EXP-107"), *extra]
    with open(f"{out}/train.log", "a") as log:
        p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                             env={**os.environ, "PYTHONUNBUFFERED": "1", **ENV})
        for line in p.stdout:
            print(line, end=""); log.write(line)
        print(preset, "exit code:", p.wait())

for preset in LADDER:
    run(preset)


In [ ]:
rows = []
for preset in LADDER:
    gp = f"{RUN}/nfm/{preset}/grades.json"
    if os.path.exists(gp):
        g = json.load(open(gp))
        rows.append((preset, g["params"]["non_embedding_active"], g["best_valid_loss"], g["throughput"]["tokens_per_s"],
                     g["throughput"]["tokens_per_s_per_chip"], g["train_minutes"], g["license"]))
print("preset | non-emb params | best valid loss | tok/s | tok/s/chip | train min | license")
for r in rows:
    print(" | ".join(str(x) for x in r))
print(f"session wall clock: {(time.time() - T_START) / 3600:.2f} h")
json.dump(rows, open(f"{RUN}/ladder_summary.json", "w"))
